<a href="https://colab.research.google.com/github/sk25469/kvern/blob/main/KVern_POC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
# Install the tokenizer library
!pip install transformers jinja2

from huggingface_hub import login
from google.colab import userdata

try:
    # This automatically grabs the token from your Colab Secrets
    token = userdata.get('HF_TOKEN')
    login(token=token)
except Exception as e:
    # If the secret isn't set, it falls back to the manual prompt
    print("Secret not found, falling back to manual login.")
    login()

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [12]:


from transformers import AutoTokenizer, AutoModelForCausalLM
import json
from dataclasses import dataclass, field
from typing import Dict, List, Optional
from google.colab import userdata
import os

In [22]:
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

# model_id = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
# Load the tokenizer
model_id = "meta-llama/Llama-3.2-1B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

# Define a sample conversation
messages = [
    {"role": "system", "content": "You are a helpful coding assistant."},
    {"role": "user", "content": "How do I build a Trie in Python?"}
]

# 1. Apply the Chat Template (Messages -> String)
templated_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(f"--- Templated Text ---\n{templated_text}\n")

# 2. Tokenize (String -> IDs)
token_ids = tokenizer.encode(templated_text)
print(f"--- Token IDs ---\n{token_ids[:20]}... (Total: {len(token_ids)})")

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

--- Templated Text ---
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 12 Apr 2026

You are a helpful coding assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>

How do I build a Trie in Python?<|eot_id|><|start_header_id|>assistant<|end_header_id|>



--- Token IDs ---
[128000, 128000, 128006, 9125, 128007, 271, 38766, 1303, 33025, 2696, 25, 6790, 220, 2366, 18, 198, 15724, 2696, 25, 220]... (Total: 52)


In [43]:
@dataclass
class MiniNode:
    token_id: int
    children: Dict[int, 'MiniNode'] = field(default_factory=dict)
    count: int = 1

class MiniTrie:
    def __init__(self):
        self.root = MiniNode(token_id=-1) # Root is a dummy node

    def insert(self, tokens: List[int]):
        current = self.root
        for tid in tokens:
            if tid in current.children:
                current.children[tid].count += 1
            else:
                current.children[tid] = MiniNode(token_id=tid)
            current = current.children[tid]

    def find_prefix(self, tokens: List[int]) -> int:
        """Returns the length of the longest matching prefix."""
        current = self.root
        match_length = 0
        for tid in tokens:
            if tid in current.children:
                match_length += 1
                current = current.children[tid]
            else:
                break
        return match_length

    def get_total_nodes(self):
        def count_nodes(node):
            return 1 + sum(count_nodes(child) for child in node.children.values())
        return count_nodes(self.root) - 1 # Minus 1 for dummy root

    def evict(self, target_node_count):
        """Removes leaf nodes with the lowest hit counts until target is met."""
        while self.get_total_nodes() > target_node_count:
            leaves = []

            # Helper to find all current leaf nodes and their parents
            def find_leaves(node, parent=None, tid=None):
                if not node.children:
                    if parent is not None:
                        leaves.append((parent, tid, node.count))
                    return
                for child_tid, child_node in node.children.items():
                    find_leaves(child_node, node, child_tid)

            find_leaves(self.root)

            if not leaves:
                break

            # Sort leaves by hit count (ascending)
            leaves.sort(key=lambda x: x[2])

            # Delete the "weakest" leaf
            parent, tid, count = leaves[0]
            del parent.children[tid]

            print(f"Evicted Token ID {tid} with {count} hits.")

    @property
    def score(self):
        # A simple cost-aware metric: How many times has this token
        # saved us from re-computing it?
        return self.count

# Let's test it!
trie = MiniTrie()

In [44]:
# TURN 1
turn1_msg = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Hello!"}
]
ids1 = tokenizer.encode(tokenizer.apply_chat_template(turn1_msg, tokenize=False))
trie.insert(ids1)
print(f"Turn 1 inserted. Length: {len(ids1)} tokens.")

# TURN 2 (History + New Message)
turn2_msg = turn1_msg + [
    {"role": "assistant", "content": "Hi there! How can I help?"},
    {"role": "user", "content": "Explain Tries."}
]
ids2 = tokenizer.encode(tokenizer.apply_chat_template(turn2_msg, tokenize=False))

# Check for prefix match
saved_tokens = trie.find_prefix(ids2)
print(f"Turn 2 total length: {len(ids2)} tokens.")
print(f"KV Cache Match: {saved_tokens} tokens saved!")
print(f"Efficiency: {(saved_tokens/len(ids2))*100:.1f}%")

# Insert Turn 2 into the Trie for the next turn
trie.insert(ids2)

Turn 1 inserted. Length: 40 tokens.
Turn 2 total length: 63 tokens.
KV Cache Match: 40 tokens saved!
Efficiency: 63.5%


In [30]:
# Try the standard attribute first
if hasattr(tokenizer, 'chat_template'):
    print("--- Found Chat Template ---")
    print(tokenizer.chat_template)
else:
    # If it's a 'Fast' tokenizer, it might be in the config
    print("--- Template not found in standard attribute, checking config ---")
    print(tokenizer.init_kwargs.get('chat_template'))

--- Found Chat Template ---
{{- bos_token }}
{%- if custom_tools is defined %}
    {%- set tools = custom_tools %}
{%- endif %}
{%- if not tools_in_user_message is defined %}
    {%- set tools_in_user_message = true %}
{%- endif %}
{%- if not date_string is defined %}
    {%- if strftime_now is defined %}
        {%- set date_string = strftime_now("%d %b %Y") %}
    {%- else %}
        {%- set date_string = "26 Jul 2024" %}
    {%- endif %}
{%- endif %}
{%- if not tools is defined %}
    {%- set tools = none %}
{%- endif %}

{#- This block extracts the system message, so we can slot it into the right place. #}
{%- if messages[0]['role'] == 'system' %}
    {%- set system_message = messages[0]['content']|trim %}
    {%- set messages = messages[1:] %}
{%- else %}
    {%- set system_message = "" %}
{%- endif %}

{#- System message #}
{{- "<|start_header_id|>system<|end_header_id|>\n\n" }}
{%- if tools is not none %}
    {{- "Environment: ipython\n" }}
{%- endif %}
{{- "Cutting Knowledge Da

In [45]:
def visualize(node, tokenizer, indent="", is_last=True):
    # Skip the dummy root node label
    if node.token_id != -1:
        # Decode the token ID back to text for readability
        token_text = tokenizer.decode([node.token_id]).replace("\n", "\\n")
        marker = "└── " if is_last else "├── "
        print(f"{indent}{marker}'{token_text}' (ID: {node.token_id}, Hits: {node.count})")
        indent += "    " if is_last else "│   "

    # Sort children to keep visualization consistent
    child_ids = sorted(node.children.keys())
    for i, tid in enumerate(child_ids):
        visualize(node.children[tid], tokenizer, indent, i == len(child_ids) - 1)

# Usage:
print("ROOT")
visualize(trie.root, tokenizer)

ROOT
└── '<|begin_of_text|>' (ID: 128000, Hits: 2)
    └── '<|begin_of_text|>' (ID: 128000, Hits: 2)
        └── '<|start_header_id|>' (ID: 128006, Hits: 2)
            └── 'system' (ID: 9125, Hits: 2)
                └── '<|end_header_id|>' (ID: 128007, Hits: 2)
                    └── '\n\n' (ID: 271, Hits: 2)
                        └── 'Cut' (ID: 38766, Hits: 2)
                            └── 'ting' (ID: 1303, Hits: 2)
                                └── ' Knowledge' (ID: 33025, Hits: 2)
                                    └── ' Date' (ID: 2696, Hits: 2)
                                        └── ':' (ID: 25, Hits: 2)
                                            └── ' December' (ID: 6790, Hits: 2)
                                                └── ' ' (ID: 220, Hits: 2)
                                                    └── '202' (ID: 2366, Hits: 2)
                                                        └── '3' (ID: 18, Hits: 2)
                                                 

In [46]:
# 1. Reset everything
trie = MiniTrie()

# 2. Simulate Turn 1
turn1_msg = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Hello!"}
]
ids1 = tokenizer.encode(tokenizer.apply_chat_template(turn1_msg, tokenize=False))
# Check match (should be 0)
print(f"Match before insert 1: {trie.find_prefix(ids1)}")
trie.insert(ids1)

# 3. Simulate Turn 2
turn2_msg = turn1_msg + [
    {"role": "assistant", "content": "Hi there! How can I help?"},
    {"role": "user", "content": "Explain Tries."}
]
ids2 = tokenizer.encode(tokenizer.apply_chat_template(turn2_msg, tokenize=False))

# Check match (should be ~40, NOT 63)
saved_tokens = trie.find_prefix(ids2)
print(f"Turn 2 total length: {len(ids2)} tokens.")
print(f"KV Cache Match: {saved_tokens} tokens saved!")
print(f"Efficiency: {(saved_tokens/len(ids2))*100:.1f}%")

Match before insert 1: 0
Turn 2 total length: 63 tokens.
KV Cache Match: 40 tokens saved!
Efficiency: 63.5%


In [47]:
for i in range(5):
    test_msg = turn1_msg + [{"role": "user", "content": f"Question {i}"}]
    trie.insert(tokenizer.encode(tokenizer.apply_chat_template(test_msg, tokenize=False)))

print("ROOT")
visualize(trie.root, tokenizer)



print(f"Nodes before eviction: {trie.get_total_nodes()}")

# Let's say our GTX 1650 only has room for 45 nodes
trie.evict(45)

print(f"Nodes after eviction: {trie.get_total_nodes()}")
print("\n--- NEW TRIE STRUCTURE ---")
visualize(trie.root, tokenizer)

ROOT
└── '<|begin_of_text|>' (ID: 128000, Hits: 6)
    └── '<|begin_of_text|>' (ID: 128000, Hits: 6)
        └── '<|start_header_id|>' (ID: 128006, Hits: 6)
            └── 'system' (ID: 9125, Hits: 6)
                └── '<|end_header_id|>' (ID: 128007, Hits: 6)
                    └── '\n\n' (ID: 271, Hits: 6)
                        └── 'Cut' (ID: 38766, Hits: 6)
                            └── 'ting' (ID: 1303, Hits: 6)
                                └── ' Knowledge' (ID: 33025, Hits: 6)
                                    └── ' Date' (ID: 2696, Hits: 6)
                                        └── ':' (ID: 25, Hits: 6)
                                            └── ' December' (ID: 6790, Hits: 6)
                                                └── ' ' (ID: 220, Hits: 6)
                                                    └── '202' (ID: 2366, Hits: 6)
                                                        └── '3' (ID: 18, Hits: 6)
                                                 